In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
    'google-generativeai>=0.8.0',
    'huggingface-hub>=0.26.0',
    'python-dotenv>=1.0.0',
    'pyyaml>=6.0',
    'requests>=2.32.0',
    'pyarrow>=16.0.0',
], check=True)

In [ ]:
import os, json, time, threading, random
from pathlib import Path
from datetime import datetime

import yaml, requests
import google.generativeai as genai
from google.api_core import exceptions as google_exceptions
from huggingface_hub import HfApi

import sys
sys.path.insert(0, '/kaggle/input/datasets/mirza176528/s2s-pipline-v2-0-2')
from shared.gemini_rate_limiter import GeminiRateLimiter

WORK_DIR        = Path('/kaggle/working')
LABELED_PATH    = WORK_DIR / 'intent_labels.jsonl'
CHECKPOINT_PATH = WORK_DIR / 'checkpoint_p2a.json'
CONFIG_DIR      = Path('/kaggle/input/datasets/mirza176528/s2s-pipline-v2-0-2/config')

BATCH_SIZE  = 50
MIN_CONF    = 0.75
SAVE_EVERY  = 5
MAX_RETRIES = 6

In [ ]:
def load_secrets():
    try:
        from kaggle_secrets import UserSecretsClient
        c = UserSecretsClient()
        secrets = {}
        for k in ['HF_TOKEN_PRIMARY','HF_TOKEN_SECONDARY','HF_TOKEN_TERTIARY',
                  'GEMINI_API_KEY_01','GEMINI_API_KEY_02','GEMINI_API_KEY_03','GEMINI_API_KEY_04',
                  'CF_WORKER_URL','CF_WORKER_SECRET']:
            try:
                v = c.get_secret(k)
                if v: secrets[k] = v
            except Exception: pass
        if secrets:
            print('[secrets] Kaggle Secrets'); return secrets
    except Exception: pass
    env_file = Path('.env')
    if env_file.exists():
        from dotenv import load_dotenv; load_dotenv(env_file)
    keys = ['HF_TOKEN_PRIMARY','HF_TOKEN_SECONDARY','HF_TOKEN_TERTIARY','GEMINI_API_KEY_01']
    missing = [k for k in keys if not os.environ.get(k)]
    if missing: raise RuntimeError(f'Missing: {missing}')
    return {k: os.environ[k] for k in keys + ['GEMINI_API_KEY_02','GEMINI_API_KEY_03','GEMINI_API_KEY_04'] if os.environ.get(k)}

SECRETS  = load_secrets()
HF_TOKEN = SECRETS['HF_TOKEN_PRIMARY']

GEMINI_KEY = next((SECRETS[k] for k in ['GEMINI_API_KEY_01','GEMINI_API_KEY_02','GEMINI_API_KEY_03','GEMINI_API_KEY_04'] if SECRETS.get(k)), None)
if not GEMINI_KEY:
    raise RuntimeError('No Gemini API key found')

genai.configure(api_key=GEMINI_KEY)
GEMINI_MODEL  = genai.GenerativeModel('gemini-2.5-flash')
RATE_LIMITER  = GeminiRateLimiter(rpm_limit=14)

with open(CONFIG_DIR / 'hf_repos.yaml') as f:
    repos_cfg = yaml.safe_load(f)
with open(CONFIG_DIR / 'intent_taxonomy.yaml') as f:
    taxonomy = yaml.safe_load(f)

STAGE0_REPO = repos_cfg['repos']['stage0_codec']['repo_id']
STAGE1_REPO = repos_cfg['repos']['stage1_ce']['repo_id']
HF_API      = HfApi(token=HF_TOKEN)
print(f'[config] stage0: {STAGE0_REPO}')
print(f'[config] stage1: {STAGE1_REPO}')

In [ ]:
def load_checkpoint():
    if CHECKPOINT_PATH.exists():
        try:
            with open(CHECKPOINT_PATH) as f: state = json.load(f)
            print(f'[checkpoint] local — labeled={state["stats"]["labeled"]}')
            return state
        except Exception: pass
    try:
        url = f'https://huggingface.co/datasets/{STAGE1_REPO}/resolve/main/checkpoint_p2a.json'
        r = requests.get(url, headers={'Authorization': f'Bearer {HF_TOKEN}'}, timeout=30)
        if r.status_code == 200:
            state = r.json()
            with open(CHECKPOINT_PATH, 'w') as f: json.dump(state, f)
            return state
    except Exception: pass
    print('[checkpoint] fresh start')
    return {'done_ids': [], 'stats': {'labeled':0,'failed':0,'low_conf':0,'no_transcript':0}, 'last_updated': None}

cp_lock = threading.Lock()

def save_checkpoint(state, upload=False):
    with cp_lock:
        state['last_updated'] = datetime.utcnow().strftime('%Y-%m-%dT%H:%M:%SZ')
        tmp = str(CHECKPOINT_PATH) + '.tmp'
        with open(tmp, 'w') as f: json.dump(state, f)
        os.replace(tmp, str(CHECKPOINT_PATH))
    if not upload: return
    for attempt in range(6):
        try:
            HF_API.upload_file(path_or_fileobj=json.dumps(state).encode(),
                path_in_repo='checkpoint_p2a.json', repo_id=STAGE1_REPO,
                repo_type='dataset', commit_message='p2a checkpoint')
            return
        except Exception: time.sleep(min(2**attempt, 60))

state    = load_checkpoint()
done_set = set(state['done_ids'])

In [ ]:
all_intents = []
for domain, intents in taxonomy.items():
    for intent in intents:
        all_intents.append(f'{domain}.{intent}' if '.' not in intent else intent)

SYSTEM_PROMPT = f"""You are an intent labeling system for Pakistani Urdu customer support conversations.
You receive a batch of transcripts and return ONLY a valid JSON array — no preamble, no markdown, no explanation.

Each element must have exactly:
- id: string
- domain: one of [restaurant, banking, healthcare, education, general]
- intent_class: one of the valid intents listed below
- intent_slots: object with key entities extracted
- sentiment: one of [neutral, concerned, frustrated, satisfied, urgent]
- requires_tool: boolean
- suggested_tool: string or null
- labeling_confidence: float 0.0–1.0

Valid intent classes:
{json.dumps(all_intents, ensure_ascii=False)}

Rules:
- Understand Urdu+English mixed transcripts together
- If transcript is too short/unclear set labeling_confidence below 0.5
- Never invent intent classes not in the list
- Return exactly one object per input, same order"""


def label_batch(batch):
    items    = [{'id': r['id'], 'transcript': r['transcript']['urdu_script']} for r in batch]
    user_msg = json.dumps(items, ensure_ascii=False)

    for attempt in range(MAX_RETRIES):
        try:
            RATE_LIMITER.acquire()
            response = GEMINI_MODEL.generate_content(
                [SYSTEM_PROMPT + '\n\n' + user_msg],
                generation_config={'response_mime_type': 'application/json'},
            )
            raw    = response.text.strip()
            labels = json.loads(raw)
            if not isinstance(labels, list): raise ValueError('Expected JSON array')
            return labels
        except json.JSONDecodeError as e:
            if attempt == MAX_RETRIES - 1: print(f'  [label] JSON parse failed: {e}'); return []
            time.sleep(5 * (attempt + 1))
        except google_exceptions.ResourceExhausted:
            wait = min(65 * (2 ** attempt), 600)
            print(f'  [label] Gemini rate limit — sleeping {wait}s')
            time.sleep(wait)
        except Exception as e:
            if attempt == MAX_RETRIES - 1: print(f'  [label] API failed: {e}'); return []
            time.sleep(min(2**attempt, 60))
    return []

In [ ]:
metadata_local = WORK_DIR / 'metadata.jsonl'

if not metadata_local.exists():
    print('[metadata] downloading sample_for_ce from HF stage0...')
    for attempt in range(6):
        try:
            url = f'https://huggingface.co/datasets/{STAGE0_REPO}/resolve/main/metadata.jsonl'
            r = requests.get(url, headers={'Authorization': f'Bearer {HF_TOKEN}'}, timeout=120, stream=True)
            r.raise_for_status()
            with open(metadata_local, 'wb') as f:
                for chunk in r.iter_content(chunk_size=65536): f.write(chunk)
            print('[metadata] downloaded')
            break
        except Exception as e:
            time.sleep(min(2**attempt, 60))

all_records = []
with open(metadata_local, encoding='utf-8') as f:
    for line in f:
        line = line.strip()
        if not line: continue
        r = json.loads(line)
        if not r.get('quality', {}).get('usable_for_ce'): continue
        if not r.get('transcript', {}).get('urdu_script', '').strip(): continue
        if not r.get('sample_for_ce', False): continue
        all_records.append(r)

pending = [r for r in all_records if r['id'] not in done_set]
print(f'[p2a] CE-ready (sample_for_ce=True): {len(all_records)}')
print(f'[p2a] already done={len(done_set)} pending={len(pending)}')

In [ ]:
batches       = [pending[i:i + BATCH_SIZE] for i in range(0, len(pending), BATCH_SIZE)]
total_batches = len(batches)
print(f'[p2a] {total_batches} batches of up to {BATCH_SIZE} records')

for batch_idx, batch in enumerate(batches):
    labels    = label_batch(batch)
    label_map = {lbl['id']: lbl for lbl in labels if isinstance(lbl, dict) and 'id' in lbl}

    for record in batch:
        seg_id = record['id']
        lbl    = label_map.get(seg_id)

        if lbl is None:
            with cp_lock:
                state['stats']['failed'] += 1
                done_set.add(seg_id); state['done_ids'].append(seg_id)
            continue

        conf = lbl.get('labeling_confidence', 0.0)
        if conf < MIN_CONF:
            with cp_lock:
                state['stats']['low_conf'] += 1
                done_set.add(seg_id); state['done_ids'].append(seg_id)
            continue

        out_record = {
            'id': seg_id,
            'audio_ref': {'segment_id': seg_id, 'audio_token_path': f'tokens/{seg_id}.npy', 'token_shape': None},
            'transcript': {
                'urdu_script':          record['transcript']['urdu_script'],
                'contains_code_switch': record['transcript']['contains_code_switch'],
            },
            'intent': {
                'domain':              lbl.get('domain', 'general'),
                'intent_class':        lbl.get('intent_class', 'general.inquiry'),
                'intent_slots':        lbl.get('intent_slots', {}),
                'sentiment':           lbl.get('sentiment', 'neutral'),
                'requires_tool':       lbl.get('requires_tool', False),
                'suggested_tool':      lbl.get('suggested_tool', None),
                'labeling_confidence': conf,
                'labeling_source':     'gemini-2.5-flash',
            },
            'intent_vector_target': None,
            'split': record.get('split', 'train'),
        }

        with open(LABELED_PATH, 'a', encoding='utf-8') as lf:
            lf.write(json.dumps(out_record, ensure_ascii=False) + '\n')

        with cp_lock:
            state['stats']['labeled'] += 1
            done_set.add(seg_id); state['done_ids'].append(seg_id)

    upload_now = (batch_idx + 1) % (SAVE_EVERY * 4) == 0
    save_checkpoint(state, upload=upload_now)

    if (batch_idx + 1) % SAVE_EVERY == 0 or batch_idx + 1 == total_batches:
        print(f'  [{batch_idx+1}/{total_batches}] labeled={state["stats"]["labeled"]} '
              f'low_conf={state["stats"]["low_conf"]} failed={state["stats"]["failed"]}')

total_labeled = sum(1 for _ in open(LABELED_PATH, encoding='utf-8')) if LABELED_PATH.exists() else 0
print(f'\n[p2a] summary: labeled={state["stats"]["labeled"]} low_conf={state["stats"]["low_conf"]} failed={state["stats"]["failed"]}')
print(f'[p2a] total in file: {total_labeled}')
save_checkpoint(state, upload=True)
print('[done] ready for p2b_encode.ipynb')